In [1]:
from pyspark.sql import SparkSession
import getpass 
username=getpass.getuser()
spark=SparkSession. \
    builder. \
    config('spark.ui.port','0'). \
    config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
    config('spark.shuffle.useOldFetchProtocol', 'true'). \
    enableHiveSupport(). \
    master('yarn'). \
    getOrCreate()

In [2]:
orders_rdd = spark.sparkContext.textFile("/public/trendytech/orders/orders_1gb.csv")

In [9]:
mapped_orders = orders_rdd.map(lambda x:(x.split(",")[2], x.split(",")[3]))

In [4]:
customers_rdd = spark.sparkContext.textFile("/public/trendytech/retail_db/customers/part-00000")

In [5]:
mapped_customers = customers_rdd.map(lambda x:(x.split(",")[0],x.split(",")[8]))

In [7]:
customers_broadcast = spark.sparkContext.broadcast(mapped_customers.collect())

In [8]:
def get_pincode(customer_id):
    try:
        return customers_broadcast.value[customer_id]
    except:
        return "-1"

In [12]:
joined_rdd = mapped_orders.map(lambda x:(get_pincode(int(x[0])),x[1]))

In [14]:
spark.stop()